## 工具调用方式
## 方式1，直接调用

In [ ]:
## 定义一个函数中文工具来使用
from langchain_core.tools import tool


@tool # 在langchain_core.ools
def get_weather(city): # 注意作为工具的函数必须有文档注释，否则langchain报错
    """ 
    获取指定城市的天气信息
    参数：
    city：城市的名称，如"上海"
    返回值：
                天气信息字符串

    """    
    return city + '晴天，温度18°C。'

In [ ]:
get_weather.invoke("beijing") # 用@tool装饰的函数有一个invoke方法，可以用它来调用函数

'beijing晴天，温度18°C。'

In [ ]:
get_weather.invoke({"city":"shanghai"}) # 还可以这么用

'shanghai晴天，温度18°C。'

### 基于模型进行调用
#### 注意：qwen2.5vl:latest模型不支持工具调用
### qwen3-vl:latest可以使用工具

In [7]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="qwen3-vl:latest",
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
    temperature=0.1
)   

tl_model = model.bind_tools([get_weather])
resp = tl_model.invoke("北京的天气如何")

if resp.tool_calls:
    print(f"AI调用工具：{resp.tool_calls}")
else:
    print(f"AI直接回答：{resp.content}")    

AI调用工具：[{'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'call_wu5w7bnm', 'type': 'tool_call'}]


## 尝试使用模型来调用OpenWeather API

## 定义我们的工具

In [ ]:
import os
from langchain_openai import ChatOpenAI
import requests
from dotenv import load_dotenv
from langchain_core.tools import tool

load_dotenv(override=True)


@tool
def query_weather_func(city="Beijing", units="metric", language="zh_cn"):
     """ 
        获取指定城市的天气信息
        参数：
        city：城市的名称，如"上海"
        api_key: OpenWeather的api key
        返回值：
                    天气信息字符串
     
     """ 
     appid = os.getenv("OpenWeather_APi_key")
     # 构建请求URL
     url = "https://api.openweathermap.org/data/2.5/weather"
     # 设置查询参数
     params = {
         "q": city,                 # 查询的城市，默认为北京
         "appid": appid,          # API密钥
         "units": units,            # 测量单位，默认为摄氏度
         "lang": language           # 输出语言，默认为简体中文
     }
     # 发送GET请求
     response = requests.get(url, params=params)
     # 检查响应状态
     if response.status_code == 200:
         # 解析响应数据
         data = response.json()
         
         return data
     
     else:
         print(f"查询失败，状态码：{response.status_code}")
         print("响应数据：", response.text)
         return {"error":"weather api 调用失败。。。"}

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="qwen3-vl:latest",
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
    temperature=0.1
)   

tl_model = model.bind_tools([query_weather_func])
resp = tl_model.invoke("北京的天气如何")

if resp.tool_calls:
    print(f"AI想调用工具：{resp.tool_calls}")
else:
    print(f"AI直接回答：{resp.content}")   

AI调用工具：[{'name': 'query_weather', 'args': {'city': '北京', 'api_key': 'your_api_key_here'}, 'id': 'call_xaxz5xeb', 'type': 'tool_call'}]


## 从message流转看工具的调用


### 参考1，不使用@tool修饰符

In [ ]:
from langchain.messages import HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from weather_tool import query_weather

# 创建模型实例
model = ChatOpenAI(
    model="qwen3-vl:latest",
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
    temperature=0.1
)   

t_model = model.bind_tools([query_weather])
msgs = [
    # HumanMessage("广州今天天气如何？"),
    HumanMessage("北京今天天气如何？"),
]

resp = t_model.invoke(msgs)
tool_calls = resp.tool_calls

for tool_call in tool_calls:
    if tool_call['name'] == 'query_weather':
        tl_resp = ToolMessage(
            content = query_weather(tool_call),
            tool_call_id=tool_call["id"],
            name = tool_call["name"]
        )
        msgs.append(tl_resp)
print("========================messages===============================")
for msg in msgs:
    msg.pretty_print()
print("========================messages===============================")
final_resp = t_model.invoke(msgs)
print(f"final response:\n{final_resp}")


========================messages===============================
================================ Human Message =================================

北京今天天气如何？
================================= Tool Message =================================
Name: query_weather

{'coord': {'lon': 109.059, 'lat': 23.8644}, 'weather': [{'id': 800, 'main': 'Clear', 'description': '晴', 'icon': '01d'}], 'base': 'stations', 'main': {'temp': 28.23, 'feels_like': 31.56, 'temp_min': 28.23, 'temp_max': 28.23, 'pressure': 1015, 'humidity': 73, 'sea_level': 1015, 'grnd_level': 990}, 'visibility': 10000, 'wind': {'speed': 1.29, 'deg': 148, 'gust': 1.57}, 'clouds': {'all': 8}, 'dt': 1789697074, 'sys': {'country': 'CN', 'sunrise': 1789684258, 'sunset': 1789728323}, 'timezone': 28800, 'id': 1809867, 'name': 'Name', 'cod': 200}
========================messages===============================
final response:
content='北京今天天气晴朗，气温28.23℃，体感温度31.56℃，风力较小（1.29米/秒），湿度73%。空气质量良好，适合户外活动。' additional_kwargs={'refusal': None} response_

## 使用@tool修饰符

In [10]:
from langchain.messages import HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI


# 创建模型实例
model = ChatOpenAI(
    model="qwen3-vl:latest",
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
    temperature=0.1
)   

t_model = model.bind_tools([query_weather_func]) # 使用我们上面定义的工具
msgs = [
    HumanMessage("上海今天天气如何？"),
    # HumanMessage("北京今天天气如何？"),
]

resp = t_model.invoke(msgs)
tool_calls = resp.tool_calls

for tool_call in tool_calls:
    if tool_call['name'] == 'query_weather_func':
        tl_resp = ToolMessage(
            content = query_weather_func.invoke(tool_call),
            tool_call_id=tool_call["id"],
            name = tool_call["name"]
        )
        msgs.append(tl_resp)
print("========================messages===============================")
for msg in msgs:
    msg.pretty_print()
print("========================messages===============================")
final_resp = t_model.invoke(msgs)
print(f"final response:\n{final_resp}")


查询失败，状态码：404
响应数据： {"cod":"404","message":"city not found"}
========================messages===============================
================================ Human Message =================================

上海今天天气如何？
================================= Tool Message =================================
Name: query_weather_func

content='{"error": "weather api 调用失败。。。"}' name='query_weather_func' tool_call_id='call_fvs8kwfr'
========================messages===============================
final response:
content='很抱歉，当前无法获取上海的天气信息。可能是网络问题或天气服务暂时不可用，建议您稍后再尝试查询。您也可以通过其他天气应用或网站获取实时天气数据。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 368, 'prompt_tokens': 227, 'total_tokens': 595, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 182}}, 'model_provider': 'openai', 'model_name': 'qwen3-vl:latest', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-327', 'finish_reason': 'stop', 'logprobs': None} id='lc_ru